# 리뷰 텍스트 전처리 - Step 2: 중복 처리 (revised)

`reviews_step1_cleaned.csv`를 입력으로 받아 같은 작성자의 다중 리뷰 작성을 식별하고 처리한다.

## 처리 전략

| Step | 조건 | 처리 |
|------|------|------|
| 0 | 작성자 = `'-'` (탈퇴/닉변) | 각 행을 고유 ID로 분리 |
| 1 | 리뷰타입 = `month` | 중복 검토 제외, 그대로 보존 |
| 2-A | 같은 옵션 + 같은 타입 + 24h 이내 + 유사도 ≥ threshold | 1개 보존, `same_option_same_type_dup` |
| 2-A' | 같은 옵션 + 같은 타입 + 24h 이내 + 유사도 < threshold | 둘 다 보존, `same_option_same_type_unique` |
| 2-B | 다른 옵션 + 같은 타입 + 24h 이내 + 유사도 ≥ threshold | 1개 보존, `multi_option_dup` |
| 2-C | 다른 옵션 + 같은 타입 + 24h 이내 + 유사도 < threshold | 둘 다 보존, `multi_option_unique` |
| 3 | 같은 옵션 + 다른 타입 + 24h 이내 + 유사도 ≥ threshold | 1개 보존, `multi_type_dup` |
| 4 | 같은 옵션 + 다른 타입 + 24h 이내 + 유사도 < threshold | 둘 다 보존, `multi_type_unique` |
| 5 | 다른 옵션 + 다른 타입 + 24h 이내 + 유사도 ≥ threshold | 1개 보존, `multi_both_dup` |
| 6 | 다른 옵션 + 다른 타입 + 24h 이내 + 유사도 < threshold | 둘 다 보존, `multi_both_unique` |
| 7 | 24h 초과 | `long_gap_review` 플래그 (재구매 가능성, 확정 불가) |

---
## 0. 데이터 로드

In [1]:
import re
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

INPUT_PATH = 'reviews_step1_cleaned.csv'
TEXT_COL   = '리뷰내용_clean'   # 임베딩/토픽 모델링용 (약한 정제)
NORM_COL   = '리뷰내용_norm'    # 중복 판정용 (강한 정규화, step1에서 생성)

df = pd.read_csv(INPUT_PATH, low_memory=False)
print(f'로드 완료: {len(df):,}건')
print(f'컬럼 수: {df.shape[1]}개')

# step1에서 생성된 컬럼 확인
required_cols = [TEXT_COL, NORM_COL, '전체_글자수', '한글_글자수']
for col in required_cols:
    exists = col in df.columns
    print(f'  {col}: {"✅" if exists else "❌ 없음 - step1 재실행 필요"}')

로드 완료: 685,042건
컬럼 수: 57개
  리뷰내용_clean: ✅
  리뷰내용_norm: ✅
  전체_글자수: ✅
  한글_글자수: ✅


In [2]:
# 작성일 datetime 변환
df['작성일'] = pd.to_datetime(df['작성일'], errors='coerce')

# 작성일 결측 확인
n_nat = df['작성일'].isna().sum()
print(f'작성일 결측: {n_nat:,}건')

# 텍스트 결측 → 빈 문자열로
df[TEXT_COL] = df[TEXT_COL].fillna('').astype(str)
df[NORM_COL] = df[NORM_COL].fillna('').astype(str)

# 전체_글자수 없으면 생성 (호환성)
if '전체_글자수' not in df.columns:
    df['전체_글자수'] = df[TEXT_COL].str.len()
    print('전체_글자수 컬럼 생성 (step1 재실행 권장)')

작성일 결측: 0건


---
## STEP 0. 탈퇴/닉변 작성자 처리

작성자가 `'-'`인 경우, 무신사 시스템상 익명 처리된 케이스이므로 각 행을 **서로 다른 사람**으로 취급한다.
→ 이후 중복 검토에서 자동으로 제외됨.

In [3]:
# '-' 작성자에 고유 ID 부여
df['작성자_norm'] = df['작성자'].astype(str)
mask_anon = df['작성자_norm'].str.strip() == "'-"

# 익명 행은 인덱스 기반 고유 ID로 변환
df.loc[mask_anon, '작성자_norm'] = (
    '_anon_' + df.loc[mask_anon].index.astype(str)
)

print(f"탈퇴/닉변 작성자: {mask_anon.sum():,}건 → 고유 ID로 변환")
print(f"고유 작성자 수: {df['작성자_norm'].nunique():,}명")

탈퇴/닉변 작성자: 3,975건 → 고유 ID로 변환
고유 작성자 수: 349,096명


---
## STEP 1. month 타입 분리

리뷰타입이 `month`인 리뷰는 한 달 사용 후 작성하는 후기로, 다른 타입(일반/스타일)과 시점·결이 다르다.
→ 중복 검토 대상에서 제외하고 그대로 보존.

In [4]:
print("리뷰타입 분포:")
print(df['리뷰타입'].value_counts(dropna=False))

mask_month = df['리뷰타입'] == 'month'
df_month = df[mask_month].copy()
df_main = df[~mask_month].copy()

print(f"\nmonth 리뷰: {len(df_month):,}건 (중복 검토 제외)")
print(f"main 리뷰: {len(df_main):,}건 (중복 검토 대상)")

리뷰타입 분포:
리뷰타입
goods         376942
photo         109292
style         104397
general        68881
month          25088
experience       442
Name: count, dtype: int64

month 리뷰: 25,088건 (중복 검토 제외)
main 리뷰: 659,954건 (중복 검토 대상)


---
## 보조 컬럼 + 하이브리드 유사도 함수

- **옵션키**: `(구매사이즈, 구매상세)` 튜플. NaN은 `None`으로 통일.
- **보존 기준**: `전체_글자수`(한글+영문+숫자 전체 길이) 기준. ④ 반영.
- **하이브리드 중복 판정**: 아래 3단계 중 하나라도 통과하면 중복으로 판정.
  1. **완전일치**: `리뷰내용_norm` 문자열이 동일
  2. **Jaccard** (character bi-gram, set 기반): 순수 집합 연산
  3. **Cosine** (character bi-gram, `use_idf=False`): TF 기반, IDF 왜곡 방지
- **`리뷰내용_norm` 사용**: step1에서 이미 강한 정규화(한글+영문+숫자만 남김)가 적용된 컬럼.
  step2에서 별도 정규화 불필요. 유사도 계산 시 `리뷰내용_norm`을 직접 입력.
- **길이별 임계값 차등 적용**: 짧은 리뷰일수록 유사도가 불안정하므로 더 보수적으로 처리.
- **세션**: 세션 첫 번째 리뷰 기준 24h 윈도우 (체이닝 방지). ② 반영.
- **중복 판정**: 모든 pair 비교 후 Union-Find로 중복 컴포넌트 구성. ⑤ 반영.

In [5]:
# 옵션키 생성: (사이즈, 상세) 튜플, NaN은 None
def make_option_key(s, d):
    s_v = s if pd.notna(s) else None
    d_v = d if pd.notna(d) else None
    return (s_v, d_v)

df_main['옵션키'] = [
    make_option_key(s, d)
    for s, d in zip(df_main['구매사이즈'], df_main['구매상세'])
]

# ④ 기준 리뷰 선정: 전체_글자수 (step1에서 이미 생성, 한글+영문+숫자 포함)
# 없으면 리뷰내용_clean 기준으로 생성 (호환성)
if '전체_글자수' not in df_main.columns:
    df_main['전체_글자수'] = df_main[TEXT_COL].str.len()
    print('전체_글자수 컬럼 생성 (step1 재실행 권장)')

# NORM_COL 결측 처리
df_main[NORM_COL] = df_main[NORM_COL].fillna('').astype(str)

print(f'옵션키 고유값: {df_main["옵션키"].nunique():,}개')
print(f'전체_글자수 통계:\n{df_main["전체_글자수"].describe()}')

옵션키 고유값: 1,570개
전체_글자수 통계:
count    659954.000000
mean         44.847950
std          30.572313
min           0.000000
25%          29.000000
50%          36.000000
75%          49.000000
max        1415.000000
Name: 전체_글자수, dtype: float64


In [ ]:
# 하이브리드 유사도 엔진 (v2 임계값 + final 구조)
# ─────────────────────────────────────────────────────────────────────
# v2에서 가져온 것:
#   - 데이터 기반 임계값 (24h 이내 페어 전수 분석 결과)
#   - Jaccard / Cosine 각각 별도 임계값
#   - char_wb 2~3-gram (단어 경계 포함, 한국어에 더 안정적)
#   - 짧은 텍스트(15자 미만): 더 높은 임계값으로 엄격하게 판단
# final에서 유지한 것:
#   - 리뷰내용_norm 직접 사용 (step1에서 이미 정규화 완료, 별도 정규화 불필요)
#   - is_duplicate() → bool 반환 (process_group 호환)
# ─────────────────────────────────────────────────────────────────────
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ── 임계값 (v2 데이터 기반 결정) ─────────────────────────────────────
# 근거: 동일 (작성자, 상품, 옵션, 리뷰타입) 그룹 내 24h 이내 페어 전수 추출
#       → 이봉 분포 확인 (0~0.1 구간 + 0.9~1.0 구간에 집중)
#       → 0.5 미만 샘플에도 유사 케이스 존재 → 단일 임계값의 한계 확인
JACCARD_THRESHOLD       = 0.7    # 일반 텍스트 Jaccard 기준
COSINE_THRESHOLD        = 0.8    # 일반 텍스트 Cosine  기준
SHORT_JACCARD_THRESHOLD = 0.9    # 짧은 텍스트 Jaccard 기준 (엄격)
SHORT_COSINE_THRESHOLD  = 0.95   # 짧은 텍스트 Cosine  기준 (엄격)
SHORT_TEXT_LIMIT        = 15     # 리뷰내용_norm 기준 글자 수 (미만이면 짧은 텍스트)

# ── 벡터라이저 (v2: char_wb 2~3-gram) ───────────────────────────────
# char_wb: 단어 경계(공백)를 패딩으로 포함 → 한국어 어미 변화에 더 안정적
vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 3), min_df=1)

# ── Jaccard 유사도 (character bi-gram, set 기반) ─────────────────────
def char_bigrams(s: str) -> set:
    return {s[i:i+2] for i in range(len(s) - 1)} if len(s) >= 2 else {s}

def jaccard_sim(s1: str, s2: str) -> float:
    """character bi-gram Jaccard. 리뷰내용_norm은 이미 정규화됨."""
    g1 = char_bigrams(str(s1 or ''))
    g2 = char_bigrams(str(s2 or ''))
    union = g1 | g2
    return round(len(g1 & g2) / len(union), 4) if union else 0.0

# ── Cosine 유사도 (char_wb TF-IDF) ──────────────────────────────────
def cosine_sim(s1: str, s2: str) -> float:
    """TF-IDF char_wb 코사인 유사도. 리뷰내용_norm은 이미 정규화됨."""
    n1 = str(s1 or '')
    n2 = str(s2 or '')
    if not n1 or not n2:
        return 0.0
    try:
        mat = vec.fit_transform([n1, n2])
        return round(float(cosine_similarity(mat[0:1], mat[1:])[0][0]), 4)
    except Exception:
        return 0.0

# ── 하이브리드 중복 판정 ─────────────────────────────────────────────
def is_duplicate(s1: str, s2: str) -> bool:
    """
    하이브리드 중복 판정 (v2 임계값 적용).
    입력: 리뷰내용_norm (step1에서 이미 정규화 완료 → 별도 정규화 불필요)

    짧은 텍스트 (15자 미만): 엄격한 임계값 적용
      → Jaccard >= 0.9 OR Cosine >= 0.95
      → 이유: 짧은 텍스트는 한두 글자 차이로 의미가 반대가 될 수 있음
              ex) '좋아요' vs '안좋아요' → 독립 처리

    일반 텍스트 (15자 이상):
      → Jaccard >= 0.7 OR Cosine >= 0.8
    """
    n1 = str(s1 or '')
    n2 = str(s2 or '')
    if not n1 or not n2:
        return False

    jac = jaccard_sim(n1, n2)
    cos = cosine_sim(n1, n2)

    # 짧은 텍스트: 더 높은 임계값 (엄격)
    if len(n1) < SHORT_TEXT_LIMIT or len(n2) < SHORT_TEXT_LIMIT:
        return (jac >= SHORT_JACCARD_THRESHOLD) or (cos >= SHORT_COSINE_THRESHOLD)

    # 일반 텍스트
    return (jac >= JACCARD_THRESHOLD) or (cos >= COSINE_THRESHOLD)

# ── 동작 확인 ────────────────────────────────────────────────────────
print('=' * 70)
print('[하이브리드 유사도 테스트 (v2 임계값, 리뷰내용_norm 기준)]')
print(f'  일반:  Jaccard >= {JACCARD_THRESHOLD} OR Cosine >= {COSINE_THRESHOLD}')
print(f'  짧은:  Jaccard >= {SHORT_JACCARD_THRESHOLD} OR Cosine >= {SHORT_COSINE_THRESHOLD}  ({SHORT_TEXT_LIMIT}자 미만)')
print('=' * 70)
print(f"  {'케이스':<32} {'Jac':>6} {'Cos':>6} {'짧은?':>5} {'중복?'}")
print(f"  {'-'*65}")

# 테스트는 리뷰내용_norm 처리된 문자열로 (공백/문장부호 이미 제거된 상태)
test_cases = [
    ('문장부호만 다름',       '정말좋아요잘입을게요',              '정말좋아요잘입을게요'),
    ('공백만 다름',           '가성비좋고핏도정사이즈',            '가성비좋고핏도정사이즈'),
    ('부분 유사',             '두께감적당하고핏좋습니다',           '두께감적당하고색감예쁘네요'),
    ('완전 다름',             '정말좋아요잘입을게요',              '사이즈가너무작아서환불'),
    ('짧은 동일',             '좋아요',                            '좋아요'),
    ('짧은 의미 반대',        '좋아요',                            '안좋아요'),
    ('짧은 유사',             '따뜻해요',                          '따뜻합니다'),
    ('일반 유사',             '제품나쁘지않네요크기적당하고따뜻한거같아요',
                              '제품잘받았습니다크기적당하고따뜻하네요굿'),
]

for name, a, b in test_cases:
    n1, n2 = str(a or ''), str(b or '')
    jac = jaccard_sim(n1, n2)
    cos = cosine_sim(n1, n2)
    dup = is_duplicate(n1, n2)
    is_short = len(n1) < SHORT_TEXT_LIMIT or len(n2) < SHORT_TEXT_LIMIT
    short_mark = '✓' if is_short else ''
    mark = '✅ 중복' if dup else '❌ 독립'
    print(f"  {name:<32} {jac:>6.3f} {cos:>6.3f} {short_mark:>5}   {mark}")

[하이브리드 유사도 테스트 (v2 임계값, 리뷰내용_norm 기준)]
  일반:  Jaccard >= 0.7 OR Cosine >= 0.8
  짧은:  Jaccard >= 0.9 OR Cosine >= 0.95  (15자 미만)
  케이스                                 Jac    Cos   짧은? 중복?
  -----------------------------------------------------------------
  문장부호만 다름                          1.000  1.000     ✓   ✅ 중복
  공백만 다름                            1.000  1.000     ✓   ✅ 중복
  부분 유사                             0.353  0.337     ✓   ❌ 독립
  완전 다름                             0.000  0.000     ✓   ❌ 독립
  짧은 동일                             1.000  1.000     ✓   ✅ 중복
  짧은 의미 반대                          0.667  0.465     ✓   ❌ 독립
  짧은 유사                             0.167  0.179     ✓   ❌ 독립
  일반 유사                             0.300  0.256         ❌ 독립


---
## 24h 세션 분리 (체이닝 방지 적용)

같은 (작성자, 상품) 안에서 작성일 순 정렬 후, **세션 첫 번째 리뷰 기준으로 24h를 초과**하면 새 세션으로 분리.

기존 방식(인접 간격만 체크)은 00시→23시→46시처럼 체이닝이 발생할 수 있어 수정함. ② 반영.

In [ ]:
# 정렬: 작성자 → 상품 → 작성일
df_main = df_main.sort_values(
    ['작성자_norm', 'goodsNo', '작성일']
).reset_index(drop=True)

# ② 체이닝 방지: 세션 첫 번째 리뷰 기준으로 24h 초과 시 새 세션 시작
# 기존: 인접 리뷰 간 시간차만 체크 → 00시->23시->46시가 같은 세션으로 묶히는 문제
# 변경: 세션 시작 시점 기준으로도 24h를 넘지 않는지 함께 확인

def assign_sessions(group: pd.DataFrame) -> pd.Series:
    times = group['작성일'].values
    sessions = []
    session_id = 0
    session_start = times[0]
    for t in times:
        diff_from_start = (t - session_start) / np.timedelta64(1, 'h')
        if diff_from_start > 24:
            session_id += 1
            session_start = t
        sessions.append(session_id)
    return pd.Series(sessions, index=group.index)

df_main['세션'] = (
    df_main.groupby(['작성자_norm', 'goodsNo'], group_keys=False)
           .apply(assign_sessions)
)

g = df_main.groupby(['작성자_norm', 'goodsNo'])
print(f"세션 부여 완료")
print(f"(작성자, 상품) 그룹 수: {g.ngroups:,}")
print(f"(작성자, 상품, 세션) 그룹 수: {df_main.groupby(['작성자_norm', 'goodsNo', '세션']).ngroups:,}")

세션 부여 완료
(작성자, 상품) 그룹 수: 469,991
(작성자, 상품, 세션) 그룹 수: 526,957


In [8]:
#pd.set_option('display.max_columns',None)
pd.options.display.max_columns = 30

In [9]:
# ③ 컬럼명 변경: is_repurchase → long_gap_review
# 주문 ID 없이 실제 재구매를 확정할 수 없으므로 더 정확한 이름으로 변경
session_per_pair = df_main.groupby(['작성자_norm', 'goodsNo'])['세션'].transform('nunique')
df_main['long_gap_review'] = session_per_pair > 1

n_long_gap_rows = df_main['long_gap_review'].sum()
n_long_gap_pairs = (session_per_pair > 1).groupby(
    [df_main['작성자_norm'], df_main['goodsNo']]
).first().sum()
print(f"long_gap_review 행 수: {n_long_gap_rows:,}건")
print(f"long_gap_review (작성자, 상품) 쌍: {n_long_gap_pairs:,}쌍")
print('※ 실제 재구매 여부는 주문 ID 없이 확정 불가. 가능성 플래그로만 활용.')

long_gap_review 행 수: 129,876건
long_gap_review (작성자, 상품) 쌍: 50,302쌍
※ 실제 재구매 여부는 주문 ID 없이 확정 불가. 가능성 플래그로만 활용.


---
## 그룹 단위 중복 처리 (Step 2 ~ 4)

`(작성자_norm, goodsNo, 세션)` 단위로 묶고:

1. 그룹 크기 = 1 → 단독 리뷰 (그대로 keep)
2. 그룹 크기 >= 2:
   - **모든 pair를 비교** (① + ⑤ 반영)
   - Union-Find로 중복 pair를 컴포넌트 단위로 묶음
   - 컴포넌트 내 `전체_글자수` 가장 많은 리뷰를 base로 선정, 나머지 drop
   - 중복 아닌 리뷰는 각각 keep

In [ ]:
def union_find_components(n: int, dup_pairs: list) -> list:
    """Union-Find로 중복 pair들을 컴포넌트(클러스터)로 묶는다."""
    parent = list(range(n))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(x, y):
        parent[find(x)] = find(y)

    for i, j in dup_pairs:
        union(i, j)

    return [find(i) for i in range(n)]


def process_group(group: pd.DataFrame) -> pd.DataFrame:
    """
    같은 (작성자, 상품, 세션) 그룹 내 다중 리뷰 처리.

    유사도 판정: is_duplicate() 하이브리드 함수 사용
      - 입력 텍스트: 리뷰내용_norm (step1에서 이미 정규화 완료)
      - 완전일치 → Jaccard → Cosine 순서로 체크
      - 길이별 threshold 차등 적용
      - 하나라도 통과하면 중복 판정
    """
    g = group.copy().reset_index(drop=False)
    n = len(g)

    if n == 1:
        g['action']   = 'keep'
        g['dup_flag'] = 'unique'
        g['is_base']  = True
        g['kept_id']  = g['리뷰번호'].iloc[0]
        return g.set_index('index')

    # 리뷰내용_norm 사용: step1에서 이미 정규화 완료 → 별도 정규화 불필요
    texts   = g[NORM_COL].tolist()
    options = g['옵션키'].tolist()
    rtypes  = g['리뷰타입'].tolist()

    # ── 모든 pair 비교 ────────────────────────────────────────────────
    dup_pairs  = []
    pair_flags = {}

    for i in range(n):
        for j in range(i + 1, n):
            same_opt = (options[i] == options[j])
            same_typ = (rtypes[i]  == rtypes[j])
            dup      = is_duplicate(texts[i], texts[j])  # 하이브리드 판정

            if same_opt and same_typ:
                if dup:
                    flag = 'same_option_same_type_dup'
                    dup_pairs.append((i, j))
                else:
                    flag = 'same_option_same_type_unique'
            elif (not same_opt) and same_typ:
                if dup:
                    flag = 'multi_option_dup'
                    dup_pairs.append((i, j))
                else:
                    flag = 'multi_option_unique'
            elif same_opt and (not same_typ):
                if dup:
                    flag = 'multi_type_dup'
                    dup_pairs.append((i, j))
                else:
                    flag = 'multi_type_unique'
            else:
                if dup:
                    flag = 'multi_both_dup'
                    dup_pairs.append((i, j))
                else:
                    flag = 'multi_both_unique'

            pair_flags[(i, j)] = flag

    # ── Union-Find로 중복 컴포넌트 구성 ──────────────────────────────
    comp_ids = union_find_components(n, dup_pairs)

    # ── 컴포넌트별 대표(base) 선정: 전체_글자수 최대 → 동률 시 작성일 빠른 것
    g_sorted = g.sort_values(['전체_글자수', '작성일'], ascending=[False, True])
    comp_base = {}
    for pos in g_sorted.index:
        cid = comp_ids[pos]
        if cid not in comp_base:
            comp_base[cid] = pos

    # ── 각 행에 action / dup_flag / is_base / kept_id 부여 ───────────
    actions  = []
    flags    = []
    is_bases = []
    kept_ids = []

    for pos in range(n):
        cid      = comp_ids[pos]
        base_pos = comp_base[cid]
        is_base  = (pos == base_pos)
        kept_id  = g.loc[base_pos, '리뷰번호']

        if is_base:
            action = 'keep'
            comp_dup_flags = [
                pair_flags[(min(i, j), max(i, j))]
                for (i, j) in dup_pairs
                if comp_ids[i] == cid
            ]
            if comp_dup_flags:
                flag = next((f for f in comp_dup_flags if f.endswith('_dup')),
                            comp_dup_flags[0])
            else:
                flag = 'unique'
        else:
            i, j = (min(pos, base_pos), max(pos, base_pos))
            flag = pair_flags.get((i, j))
            if flag is None:
                comp_dup_flags = [
                    pair_flags[(min(a, b), max(a, b))]
                    for (a, b) in dup_pairs
                    if comp_ids[a] == cid
                ]
                flag = comp_dup_flags[0] if comp_dup_flags else 'same_option_same_type_dup'
            action = 'drop' if flag.endswith('_dup') else 'keep'

        actions.append(action)
        flags.append(flag)
        is_bases.append(is_base)
        kept_ids.append(kept_id)

# keep된 리뷰끼리 추가 중복 체크
    keep_indices = [pos for pos in range(n) if actions[pos] == 'keep']

    if len(keep_indices) > 1:
        for i in range(len(keep_indices)):
            for j in range(i + 1, len(keep_indices)):
                pi, pj = keep_indices[i], keep_indices[j]
                if actions[pi] == 'drop' or actions[pj] == 'drop':
                    continue
                if is_duplicate(texts[pi], texts[pj]):
                    if g.loc[pi, '전체_글자수'] >= g.loc[pj, '전체_글자수']:
                        actions[pj] = 'drop'
                    else:
                        actions[pi] = 'drop'

    g['action']   = actions
    g['dup_flag'] = flags
    g['is_base']  = is_bases
    g['kept_id']  = kept_ids

    return g.set_index('index')

In [11]:
# 그룹 사이즈 사전 확인
group_sizes = df_main.groupby(['작성자_norm', 'goodsNo', '세션']).size()
print(f"전체 (작성자, 상품, 세션) 그룹 수: {len(group_sizes):,}")
print(f"  단일 리뷰 그룹: {(group_sizes == 1).sum():,}")
print(f"  다중 리뷰 그룹(≥2): {(group_sizes >= 2).sum():,}")
print(f"\n다중 리뷰 그룹 크기 분포:")
print(group_sizes[group_sizes >= 2].value_counts().sort_index().head(10))

전체 (작성자, 상품, 세션) 그룹 수: 526,957
  단일 리뷰 그룹: 425,413
  다중 리뷰 그룹(≥2): 101,544

다중 리뷰 그룹 크기 분포:
2    72027
3    28358
4      735
5      101
6      309
7        5
8        2
9        7
Name: count, dtype: int64


In [12]:
# 단일 그룹 / 다중 그룹 분리 처리 (성능 최적화)
df_main = df_main.merge(
    group_sizes.rename('그룹크기').reset_index(),
    on=['작성자_norm', 'goodsNo', '세션'],
    how='left'
)

mask_single = df_main['그룹크기'] == 1
df_single = df_main[mask_single].copy()
df_multi  = df_main[~mask_single].copy()

print(f"단일 그룹 행: {len(df_single):,}")
print(f"다중 그룹 행: {len(df_multi):,}")

# 단일 그룹: 일괄 처리
df_single['action']   = 'keep'
df_single['dup_flag'] = 'unique'
df_single['is_base']  = True
df_single['kept_id']  = df_single['리뷰번호']

# 다중 그룹: process_group 적용
tqdm.pandas(desc='다중 그룹 처리')
df_multi_processed = (
    df_multi.groupby(['작성자_norm', 'goodsNo', '세션'], group_keys=False, sort=False)
            .progress_apply(process_group)
)

# pandas 버전 호환: groupby 키 컬럼이 결과에서 빠진 경우(pandas 3.0+) 복구
for col in ['작성자_norm', 'goodsNo', '세션']:
    if col not in df_multi_processed.columns:
        df_multi_processed[col] = df_multi.loc[df_multi_processed.index, col]

# 합치기
df_main_processed = pd.concat(
    [df_single, df_multi_processed], ignore_index=True
).sort_values(['작성자_norm', 'goodsNo', '작성일']).reset_index(drop=True)

print(f"\n처리 완료: {len(df_main_processed):,}건")

단일 그룹 행: 425,413
다중 그룹 행: 234,541


다중 그룹 처리:   0%|          | 0/101544 [00:00<?, ?it/s]


처리 완료: 659,954건


---
## 처리 결과 통계

In [13]:
print('[action 분포]')
print(df_main_processed['action'].value_counts())
print(f'\n[dup_flag 분포]')
print(df_main_processed['dup_flag'].value_counts(dropna=False))
print(f'\n[long_gap_review 분포]')
print(df_main_processed['long_gap_review'].value_counts())

[action 분포]
action
keep    598579
drop     61375
Name: count, dtype: int64

[dup_flag 분포]
dup_flag
unique                       552577
multi_type_dup               102099
multi_option_dup               3978
multi_both_dup                  713
same_option_same_type_dup       519
multi_type_unique                65
multi_both_unique                 2
multi_option_unique               1
Name: count, dtype: int64

[long_gap_review 분포]
long_gap_review
False    530078
True     129876
Name: count, dtype: int64


In [14]:
# 샘플 확인: drop된 리뷰 vs 보존된 기준 리뷰 페어
print('[multi_option_dup 샘플]')
sample_dup = df_main_processed[
    (df_main_processed['dup_flag'] == 'multi_option_dup') &
    (df_main_processed['action'] == 'drop')
].head(5)
for _, row in sample_dup.iterrows():
    base = df_main_processed[df_main_processed['리뷰번호'] == row['kept_id']].iloc[0]
    print(f"\n  [기준] {row['kept_id']} | 옵션={base['옵션키']} | 전체_글자수={base['전체_글자수']}")
    print(f"     → {base[TEXT_COL][:80]}")
    print(f"  [drop] {row['리뷰번호']} | 옵션={row['옵션키']} | 전체_글자수={row['전체_글자수']}")
    print(f"     → {row[TEXT_COL][:80]}")

[multi_option_dup 샘플]

  [기준] 15343006 | 옵션=('M', '셔츠딥그레이') | 전체_글자수=91
     → 셋업에 벌룬핏이라 색상별로 구매했습니다 두께감은 일반청자켓 셋업 정도로 생각하시면 되고 한벌 또는 셔츠와 팬츠를 별로도 입을수있어 환절기에 적합
  [drop] 15342958 | 옵션=('M', '셔츠라이트베이지') | 전체_글자수=91
     → 셋업에 벌룬핏이라 색상별로 구매했습니다 두께감은 일반청자켓 셋업 정도로 생각하시면 되고 한벌 또는 셔츠와 팬츠를 별로도 입을수있어 환절기에 적합

  [기준] 15343006 | 옵션=('M', '셔츠딥그레이') | 전체_글자수=91
     → 셋업에 벌룬핏이라 색상별로 구매했습니다 두께감은 일반청자켓 셋업 정도로 생각하시면 되고 한벌 또는 셔츠와 팬츠를 별로도 입을수있어 환절기에 적합
  [drop] 15342623 | 옵션=('M', '셔츠블랙') | 전체_글자수=91
     → 셋업에 벌룬핏이라 색상별로 구매했습니다 두께감은 일반청자켓 셋업 정도로 생각하시면 되고 한벌 또는 셔츠와 팬츠를 별로도 입을수있어 환절기에 적합

  [기준] 42693190 | 옵션=('2XL', '베이지') | 전체_글자수=56
     → 옷감이 가벼워 여름에 시원하게 입을수 있겠네요뒷주머니가 사진에 안보였는데 진짜로 없어 좀 아쉽네요;;
  [drop] 42693216 | 옵션=('M', '베이지') | 전체_글자수=56
     → 옷감이 가벼워 여름에 시원하게 입을수 있겠네요뒷주머니가 사진에 안보였는데 진짜로 없어 좀 아쉽네요;;

  [기준] 44590746 | 옵션=('XL', '차콜') | 전체_글자수=36
     → 다른색들도 다 맘에들어서 더 샀는데 역시 맘에드네요 감사합니다ㅎㅎ
  [drop] 44590756 | 옵션=('XL', '블랙') | 전체_글자수=36
     → 다른색들도 다 맘에들어서 더 샀는데 역시 맘에드네요 감사합니다ㅎㅎ

  [기준] 44590746

In [15]:
print('[multi_option_unique 샘플 - 둘 다 보존되는 케이스]')
companions = df_main_processed[
    (df_main_processed['dup_flag'] == 'multi_option_unique') &
    (~df_main_processed['is_base'])
].head(5)
for _, row in companions.iterrows():
    base = df_main_processed[df_main_processed['리뷰번호'] == row['kept_id']].iloc[0]
    print(f"\n  [기준]      {row['kept_id']} | 옵션={base['옵션키']} | 전체_글자수={base['전체_글자수']}")
    print(f"     → {base[TEXT_COL][:100]}")
    print(f"  [같이 보존] {row['리뷰번호']} | 옵션={row['옵션키']} | 전체_글자수={row['전체_글자수']}")
    print(f"     → {row[TEXT_COL][:100]}")

[multi_option_unique 샘플 - 둘 다 보존되는 케이스]

  [기준]      21752415 | 옵션=('XL', '애쉬블루') | 전체_글자수=57
     → 이것도 색깔별로 다 사게됬어요 적당한 두께감 박시한 크기 다양한 색상 저렴한 가격 가격대비 높은 퀼리티
  [같이 보존] 21752459 | 옵션=('XL', '망고') | 전체_글자수=55
     → 이것도 색깔별로 다 사게됬어요 적당한 두께감 박시한 크기 다양한 색상 저렴한 가격 여름엔 좀 더울듯


---
## month 리뷰 통합 + 저장

- `reviews_step2_dedup.csv`: 최종 보존 리뷰 (Step 3 임베딩 입력)
- `reviews_step2_dropped.csv`: drop된 리뷰 (검증용 보관)

In [16]:
# month 리뷰는 그대로 keep, 플래그만 추가
df_month['action']          = 'keep'
df_month['dup_flag']        = 'month_excluded'
df_month['is_base']         = True
df_month['kept_id']         = df_month['리뷰번호']
df_month['long_gap_review'] = False
df_month['세션']             = 0
df_month['그룹크기']         = 1
df_month['옵션키'] = [
    make_option_key(s, d)
    for s, d in zip(df_month['구매사이즈'], df_month['구매상세'])
]
if '전체_글자수' not in df_month.columns:
    df_month['전체_글자수'] = df_month['리뷰내용_clean'].str.len()

# 컬럼 정렬을 df_main_processed에 맞춤
df_month = df_month[df_main_processed.columns]

# 통합
df_final = pd.concat([df_main_processed, df_month], ignore_index=True)
print(f'통합 후: {len(df_final):,}건')

통합 후: 685,042건


In [ ]:
# 최종 저장: keep만
df_keep = df_final[df_final['action'] == 'keep'].copy()
df_drop = df_final[df_final['action'] == 'drop'].copy()

print(f"최종 보존: {len(df_keep):,}건 ({len(df_keep)/len(df_final)*100:.2f}%)")
print(f"제거된 중복: {len(df_drop):,}건 ({len(df_drop)/len(df_final)*100:.2f}%)")

print(f"\n[보존 리뷰 dup_flag 분포]")
print(df_keep['dup_flag'].value_counts())

print(f"\n[제거된 리뷰 dup_flag 분포]")
print(df_drop['dup_flag'].value_counts())

최종 보존: 623,667건 (91.04%)
제거된 중복: 61,375건 (8.96%)

[보존 리뷰 dup_flag 분포]
dup_flag
unique                       552577
multi_type_dup                43841
month_excluded                25088
multi_option_dup               1794
same_option_same_type_dup       233
multi_both_dup                   66
multi_type_unique                65
multi_both_unique                 2
multi_option_unique               1
Name: count, dtype: int64

[제거된 리뷰 dup_flag 분포]
dup_flag
multi_type_dup               58258
multi_option_dup              2184
multi_both_dup                 647
same_option_same_type_dup      286
Name: count, dtype: int64


#### 플래그 전체 목록

***중복 판정 기준**:

_dup  
- (Jaccard >= 0.7 OR Cosine >= 0.8, 중복)   ← 일반 텍스트 (15자 이상)
- (Jaccard >= 0.9 OR Cosine >= 0.95, 중복)   ← 짧은 텍스트 (15자 미만)

_unique (위 조건 미충족, 다른 내용)


| 플래그 | 조건 | action |
|--------|------|--------|
| `unique` | 그룹 내 단독 리뷰 | keep |
| `month_excluded` | 한 달 사용 후기 | keep |
| `same_option_same_type_dup` | 같은 옵션+타입 + 중복 판정* | drop |
| `same_option_same_type_unique` | 같은 옵션+타입 + 중복 아님* | keep |
| `multi_option_dup` | 다른 옵션+같은 타입 + 중복 판정* | drop |
| `multi_option_unique` | 다른 옵션+같은 타입 + 중복 아님* | keep |
| `multi_type_dup` | 같은 옵션+다른 타입 + 중복 판정* | drop |
| `multi_type_unique` | 같은 옵션+다른 타입 + 중복 아님* | keep |
| `multi_both_dup` | 다른 옵션+다른 타입 + 중복 판정* | drop |
| `multi_both_unique` | 다른 옵션+다른 타입 + 중복 아님* | keep |

**변경 사항 요약**
- ① `same_option_same_type`에 유사도 체크 추가 + `_unique` 플래그 신규
- ② 세션 분리: 인접 간격 기준 → 세션 시작 기준 24h 윈도우 (체이닝 방지)
- ③ `is_repurchase` → `long_gap_review`
- ④ base 선정: `한글_글자수` → `전체_글자수`
- ⑤ base vs 나머지 → 모든 pair 비교 + Union-Find 컴포넌트

In [18]:
# CSV 저장
OUTPUT_KEEP = "reviews_step2_dedup.csv"
OUTPUT_DROP = "reviews_step2_dropped.csv"

df_keep.to_csv(OUTPUT_KEEP, index=False)
df_drop.to_csv(OUTPUT_DROP, index=False)

print(f"저장 완료:")
print(f"  - {OUTPUT_KEEP} ({len(df_keep):,}건)")
print(f"  - {OUTPUT_DROP} ({len(df_drop):,}건)")

저장 완료:
  - reviews_step2_dedup.csv (623,667건)
  - reviews_step2_dropped.csv (61,375건)


---
## (참고) 검증용 분석

전체 처리 결과를 한눈에 점검하기 위한 통계.
실제 분석에는 사용하지 않으므로 필요 시 셀 단위로 참고.

In [19]:
# 옵션 다중 구매 케이스 통계
multi_opt_pairs = df_keep[df_keep['dup_flag'].isin([
    'multi_option_unique', 'multi_option_dup'
])]
print(f'옵션 다중 구매 관련 보존된 리뷰: {len(multi_opt_pairs):,}건')
print(f'  - base: {multi_opt_pairs["is_base"].sum():,}건')
print(f'  - 동반자(unique 케이스): {(~multi_opt_pairs["is_base"]).sum():,}건')

# long_gap_review 통계
long_gap_keep = df_keep[df_keep['long_gap_review']]
print(f'\nlong_gap_review 케이스 보존 리뷰: {len(long_gap_keep):,}건')
print(f'long_gap_review (작성자, 상품) 쌍: {long_gap_keep.groupby(["작성자_norm", "goodsNo"]).ngroups:,}쌍')
print('※ 실제 재구매 여부는 주문 ID 없이 확정 불가.')

if '브랜드' in df_keep.columns and '카테고리' in df_keep.columns:
    print(f'\n[브랜드 x 카테고리별 보존 리뷰 수]')
    print(df_keep.groupby(['브랜드', '카테고리']).size().unstack(fill_value=0))

옵션 다중 구매 관련 보존된 리뷰: 1,795건
  - base: 1,794건
  - 동반자(unique 케이스): 1건

long_gap_review 케이스 보존 리뷰: 120,652건
long_gap_review (작성자, 상품) 쌍: 50,302쌍
※ 실제 재구매 여부는 주문 ID 없이 확정 불가.


In [20]:
# ── 플래그별 샘플 검증 ────────────────────────────────────────────────────
# 같은 그룹(작성자, 상품, 세션)의 모든 멤버를 함께 출력해서
# 어떤 리뷰가 왜 keep/drop 됐는지 맥락과 함께 확인할 수 있음

import pandas as pd
pd.set_option('display.max_colwidth', None)

SAMPLE_N     = 3    # 플래그별 출력할 그룹 수
TEXT_PREVIEW = 150  # 텍스트 미리보기 길이
SEP_WIDE     = '=' * 100
SEP_NARROW   = '-' * 100

def print_group_members(group_rows: pd.DataFrame, source_df: pd.DataFrame):
    """그룹 내 모든 멤버를 keep/drop 구분하여 출력."""
    # 그룹 키 정보
    sample = group_rows.iloc[0]
    print(f"  작성자={sample['작성자_norm']} | 상품={sample['goodsNo']} | 세션={sample['세션']}")
    print(f"  그룹 크기={len(group_rows)}명  /  ", end='')
    print(f"keep={( group_rows['action']=='keep').sum()}건, drop={(group_rows['action']=='drop').sum()}건")
    print(SEP_NARROW)

    # keep 먼저, drop 나중에 정렬
    ordered = pd.concat([
        group_rows[group_rows['action'] == 'keep'].sort_values('전체_글자수', ascending=False),
        group_rows[group_rows['action'] == 'drop'].sort_values('전체_글자수', ascending=False),
    ])

    for mi, (_, row) in enumerate(ordered.iterrows(), 1):
        action_label = '✅ KEEP' if row['action'] == 'keep' else '❌ DROP'
        base_label   = ' [base]' if row['is_base'] else '       '
        print(f"  {action_label}{base_label}  리뷰번호={row['리뷰번호']}")
        print(f"           dup_flag={row['dup_flag']}")
        print(f"           옵션={row['옵션키']} | 타입={row['리뷰타입']} | 전체_글자수={row['전체_글자수']} | 작성일={str(row['작성일'])[:16]}")
        print(f"           텍스트: {str(row['리뷰내용_clean'])[:TEXT_PREVIEW]}")
        if mi < len(ordered):
            print()


def show_flag_samples(flag_name: str, source_df: pd.DataFrame, n: int = SAMPLE_N):
    """
    특정 dup_flag를 가진 그룹을 n개 샘플링해서
    그룹 내 모든 멤버(keep + drop)를 함께 출력.
    """
    print(f"\n{SEP_WIDE}")
    print(f"  dup_flag = [{flag_name}]")
    print(SEP_WIDE)

    # 이 플래그가 존재하는 그룹 키 추출 (동반자 기준으로 샘플링)
    target = source_df[source_df['dup_flag'] == flag_name]
    if len(target) == 0:
        print('  (해당 플래그 없음)')
        return

    # 그룹 키 유니크 추출 후 샘플링
    group_keys = (
        target[['작성자_norm', 'goodsNo', '세션']]
        .drop_duplicates()
        .sample(n=min(n, len(target[['작성자_norm', 'goodsNo', '세션']].drop_duplicates())), random_state=42)
    )

    for gi, (_, key) in enumerate(group_keys.iterrows(), 1):
        print(f"\n  ── 그룹 {gi} ──")
        group_rows = source_df[
            (source_df['작성자_norm'] == key['작성자_norm']) &
            (source_df['goodsNo']     == key['goodsNo']) &
            (source_df['세션']        == key['세션'])
        ].copy()
        print_group_members(group_rows, source_df)


def show_solo_samples(flag_name: str, source_df: pd.DataFrame, n: int = SAMPLE_N):
    """unique, month_excluded 같은 단독 플래그용 출력."""
    print(f"\n{SEP_WIDE}")
    print(f"  dup_flag = [{flag_name}]")
    print(SEP_WIDE)

    pool = source_df[source_df['dup_flag'] == flag_name]
    if len(pool) == 0:
        print('  (해당 플래그 없음)')
        return

    sampled = pool.sample(n=min(n, len(pool)), random_state=42)
    for k, (_, row) in enumerate(sampled.iterrows(), 1):
        print(f"\n  ── 샘플 {k} ──")
        print(f"  리뷰번호={row['리뷰번호']} | dup_flag={row['dup_flag']} | action={row['action']}")
        print(f"  옵션={row['옵션키']} | 타입={row['리뷰타입']} | 전체_글자수={row['전체_글자수']}")
        print(f"  텍스트: {str(row['리뷰내용_clean'])[:TEXT_PREVIEW]}")


# ── 실행: 플래그별 그룹 샘플 출력 ───────────────────────────────────────
source = df_final

# _dup 플래그: keep(base) + drop(동반자) 함께 출력
for flag in [
    'same_option_same_type_dup',
    'multi_option_dup',
    'multi_type_dup',
    'multi_both_dup',
]:
    show_flag_samples(flag, source)

# _unique 플래그: 둘 다 keep인 케이스
for flag in [
    'same_option_same_type_unique',
    'multi_option_unique',
    'multi_type_unique',
    'multi_both_unique',
]:
    show_flag_samples(flag, source)

# 단독 플래그
for flag in ['unique', 'month_excluded']:
    show_solo_samples(flag, source)


  dup_flag = [same_option_same_type_dup]

  ── 그룹 1 ──
  작성자=ToRi. | 상품=552760 | 세션=0
  그룹 크기=6명  /  keep=1건, drop=5건
----------------------------------------------------------------------------------------------------
  ✅ KEEP [base]  리뷰번호=15515152
           dup_flag=multi_type_dup
           옵션=('S', '화이트') | 타입=style | 전체_글자수=69 | 작성일=2021-03-12 16:20
           텍스트: 레이어드하기에 너무 길지않아 S사이즈가 적당한 것 같고 평소에 100-105사이즈 옷을 입는데 가슴둘레가 충분해서 좋았어요.

  ❌ DROP         리뷰번호=15515167
           dup_flag=multi_type_dup
           옵션=('S', '화이트') | 타입=photo | 전체_글자수=69 | 작성일=2021-03-12 16:21
           텍스트: 레이어드하기에 너무 길지않아 S사이즈가 적당한 것 같고 평소에 100-105사이즈 옷을 입는데 가슴둘레가 충분해서 좋았어요.

  ❌ DROP         리뷰번호=15515168
           dup_flag=multi_type_dup
           옵션=('S', '화이트') | 타입=goods | 전체_글자수=69 | 작성일=2021-03-12 16:21
           텍스트: 레이어드하기에 너무 길지않아 S사이즈가 적당한 것 같고 평소에 100-105사이즈 옷을 입는데 가슴둘레가 충분해서 좋았어요.

  ❌ DROP         리뷰번호=15515179
           dup_flag=same_option_same_type_dup
           옵션=('S', '화이트'

In [21]:
# 그룹 크기별 분포 확인
print("[(작성자, 상품, 세션) 그룹 크기 분포]")
print(df_final[df_final['리뷰타입'] != 'month']
      .groupby(['작성자_norm', 'goodsNo', '세션']).size()
      .value_counts().sort_index())

[(작성자, 상품, 세션) 그룹 크기 분포]
1    425413
2     72027
3     28358
4       735
5       101
6       309
7         5
8         2
9         7
Name: count, dtype: int64


In [ ]:
# 그룹 크기 ≥ 3인 케이스만 추출해서 그룹 단위로 모든 멤버 표시

LARGE_GROUP_SAMPLE_N = 5  # 출력할 그룹 수

def print_large_group_samples(source_df, min_size=3, n=LARGE_GROUP_SAMPLE_N):
    """그룹 사이즈 ≥ min_size 인 케이스를 그룹 단위로 출력. 모든 멤버를 한 번에 보여줌."""
    
    # month는 별도 처리되므로 제외
    df_check = source_df[source_df['리뷰타입'] != 'month'].copy()
    
    # 그룹 크기 계산
    sizes = df_check.groupby(['작성자_norm', 'goodsNo', '세션']).size()
    large_groups = sizes[sizes >= min_size]
    
    if len(large_groups) == 0:
        print(f"\n그룹 크기 ≥ {min_size}인 케이스 없음")
        return
    
    print(f"\n{'='*95}")
    print(f"[그룹 크기 ≥ {min_size} 케이스: 총 {len(large_groups):,}그룹]")
    print('='*95)
    
    # 그룹 크기 분포
    print("\n그룹 크기별 개수:")
    print(large_groups.value_counts().sort_index().to_string())
    
    # 무작위 n개 그룹 선택
    sample_keys = large_groups.sample(n=min(n, len(large_groups)), 
                                       random_state=42).index.tolist()
    
    for gi, key in enumerate(sample_keys, 1):
        author, goods, sess = key
        members = df_check[
            (df_check['작성자_norm'] == author) &
            (df_check['goodsNo'] == goods) &
            (df_check['세션'] == sess)
        ].sort_values(['is_base', '전체_글자수'], ascending=[False, False])
        
        # 이 그룹 안의 dup_flag 종류 한눈에
        flag_summary = members['dup_flag'].value_counts().to_dict()
        
        print(f"\n{'─'*95}")
        print(f"그룹 {gi}: 작성자={author} | 상품={goods} | 세션={sess} | 멤버 {len(members)}명")
        print(f"  플래그 구성: {flag_summary}")
        print(f"  base 리뷰번호: {members[members['is_base']]['리뷰번호'].iloc[0]}")
        print(f"{'─'*95}")
        
        for mi, (_, m) in enumerate(members.iterrows(), 1):
            role = "★ base" if m['is_base'] else "  동반자"
            print(f"\n  [{role}] 멤버 {mi}")
            print(f"    리뷰번호={m['리뷰번호']} | dup_flag={m['dup_flag']} | "
                  f"is_base={m['is_base']} | action={m['action']}")
            print(f"    옵션={m['옵션키']} | 타입={m['리뷰타입']} | "
                  f"전체_글자수={m['전체_글자수']} | 작성일={m['작성일']}")
            print(f"    → {str(m[TEXT_COL])[:TEXT_PREVIEW]}")


# 그룹 크기별로 따로 보기
print_large_group_samples(source, min_size=3, n=5)   # 3명 이상 그룹 5개
print_large_group_samples(source, min_size=4, n=3)   # 4명 이상 그룹 3개
print_large_group_samples(source, min_size=5, n=2)   # 5명 이상 그룹 2개 (있다면)


[그룹 크기 ≥ 3 케이스: 총 29,517그룹]

그룹 크기별 개수:
3    28358
4      735
5      101
6      309
7        5
8        2
9        7

───────────────────────────────────────────────────────────────────────────────────────────────
그룹 1: 작성자=배택환 | 상품=2085369 | 세션=0 | 멤버 3명
  플래그 구성: {'unique': 3}
  base 리뷰번호: 32055090
───────────────────────────────────────────────────────────────────────────────────────────────

  [★ base] 멤버 1
    리뷰번호=32055090 | dup_flag=unique | is_base=True | action=keep
    옵션=('L', '네이비') | 타입=style | 한글_글자수=36 | 작성일=2022-09-05 18:11:03
    → 전체적으로 폼이 큽니다 팔부분에 시보리가 강해서 팔 기장이 길어도 이쁘게 떨어지네요

  [★ base] 멤버 2
    리뷰번호=32054850 | dup_flag=unique | is_base=True | action=keep
    옵션=('L', '네이비') | 타입=photo | 한글_글자수=24 | 작성일=2022-09-05 18:04:25
    → 상당히 오버한 맨투맨 입니다 가성비 좋게 잘 입을 것 같네요

  [★ base] 멤버 3
    리뷰번호=32054879 | dup_flag=unique | is_base=True | action=keep
    옵션=('L', '네이비') | 타입=goods | 한글_글자수=23 | 작성일=2022-09-05 18:05:33
    → 가성비 좋은 맨투맨 입니다 가을에 잘 입고 다닐 것 같아요

───────────────